In [36]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix

# This function remains unchanged as its inputs are still the same.
def extract_summary_tables_per_stance(final_lr_results_for_summary, combined_features_df, quartile_stats):
    """
    Generates stance-specific summary tables from fully processed data.
    
    Args:
        final_lr_results_for_summary (dict): Dict of final LR results to be summarized.
        combined_features_df (pd.DataFrame): A DataFrame containing the merged LR and significance info.
        quartile_stats (dict): Quartile misclassification stats.
    """
    summary_tables = {}
    for stance_name, coef_df in final_lr_results_for_summary.items():
        if coef_df.empty:
            continue
        
        # Get the detailed info for features in this stance from the combined dataframe
        feature_details = combined_features_df[
            combined_features_df['stance'] == stance_name
        ]
        
        rows = []
        for _, row in coef_df.iterrows():
            feature = row["feature"]
            coef = row["coefficient"]
            impact = "↑ Misclass" if coef > 0 else "↓ Misclass"

            # MODIFIED: Get significance info directly from the 'feature_details' DataFrame
            # No more redundant lookups!
            detail_row = feature_details[feature_details["feature"] == feature].iloc[0]
            sig = detail_row.get("significant", "N/A")
            eff_type = detail_row.get("preferred_effect", "-")
            eff_strength = detail_row.get("preferred_strength", "")
            eff_str = f"{eff_type} ({eff_strength})" if eff_strength else eff_type

            # Quartile info (this part remains the same)
            quart = quartile_stats.get(feature, {})
            max_rate = quart.get("max_misclass_rate", None)
            max_bin = quart.get("max_bin", "-")
            max_stance = quart.get("stance", "-")
            if max_rate is not None:
                max_info = f"{max_rate:.1f}% ({max_stance}: {max_bin})" if stance_name == "All Stances" else f"{max_rate:.1f}% ({max_bin})"
            else:
                max_info = "-"

            row_dict = {
                "Feature": feature,
                "LR Coef": f"{coef:+.3f}",
                "Significant": sig,
                "Effect Size": eff_str,
                "Impact": impact,
                "Max Misclass Rate": max_info
            }
            if stance_name == "All Stances":
                row_dict["Stance"] = max_stance
            rows.append(row_dict)

        df_summary = pd.DataFrame(rows)
        summary_tables[stance_name] = df_summary

    summary_tables_ordered = {
        "All Stances": summary_tables.get("All Stances", pd.DataFrame()),
        "AGAINST": summary_tables.get("AGAINST", pd.DataFrame()),
        "FAVOR": summary_tables.get("FAVOR", pd.DataFrame()),
        "NONE": summary_tables.get("NONE", pd.DataFrame())
    }

    return {k: v for k, v in summary_tables_ordered.items() if not v.empty}


# MODIFIED: Added parameters for flexible experiment control.
def analyze_lr_significance_quartiles_with_filter_and_summary(
        correct_path, misclassified_path, significance_df,
        dataset, source,
        feature_cols=None, quantiles=4,
        coef_direction='positive',
        coef_threshold=0.1,
        significance_filter='yes'):

    # Create a unique directory name for this experiment run
    experiment_name = f"{source}_{dataset}_coef-{coef_direction}_sig-{significance_filter}"
    output_dir = f"paper_plots_v3/{coef_threshold}_threshold_experiment/{experiment_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    summary_lines = []
    summary_lines.append(f"Running experiment: {experiment_name}\n")
    summary_lines.append(f"I am using {source} setting and {dataset} dataset \n")

    df_correct = pd.read_csv(correct_path)
    df_mis = pd.read_csv(misclassified_path)
    df_correct["label"] = "Correct"
    df_mis["label"] = "Misclassified"
    df_all = pd.concat([df_correct, df_mis], ignore_index=True)
    df_all["label_bin"] = df_all["label"].map({"Correct": 0, "Misclassified": 1})

    non_feature_cols = ['target', 'text', 'stance', 'label', 'label_bin']
    if feature_cols is None:
        feature_cols = [c for c in df_all.columns
                        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df_all[c])]

    stances = [("All Stances", None), ("AGAINST", 0), ("FAVOR", 1), ("NONE", 2)]
    lr_results = {}
    summary_lines.append("=== Logistic Regression Summary ===\n")
    
    for stance_name, stance_val in stances:
        df = df_all.copy()
        if stance_val is not None:
            df = df[df["stance"] == stance_val]
        if df.empty:
            summary_lines.append(f"[SKIP] No data for stance {stance_name}\n")
            continue

        df_model = df[feature_cols + ["label_bin"]].dropna()
        X = df_model[feature_cols].values
        y = df_model["label_bin"].values
        X_scaled = StandardScaler().fit_transform(X)
        model = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000)
        cv_score = np.mean(cross_val_score(model, X_scaled, y, cv=5, scoring="accuracy"))
        model.fit(X_scaled, y)
        y_pred = model.predict(X_scaled)
        cm = confusion_matrix(y, y_pred, labels=[0, 1])  # 0 = Correct, 1 = Misclassified
        summary_lines.append(f"Confusion matrix ({stance_name}):\n")
        summary_lines.append(f"{cm}\n\n")

        fig_cm, ax_cm = plt.subplots(figsize=(2.2, 2.2))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            xticklabels=["Correct", "Miscls"],
            yticklabels=["Correct", "Miscls"],
            ax=ax_cm,
        )
        ax_cm.set_xlabel("Predicted")
        ax_cm.set_ylabel("True")
        ax_cm.set_title(stance_name, fontsize=9)
        plt.tight_layout()
        cm_path = f"{output_dir}/cm_{stance_name.replace(' ', '_').lower()}.png"
        plt.savefig(cm_path, dpi=300, bbox_inches="tight")
        plt.close()
        

        coefs = model.coef_[0]
        # MODIFIED: Create a full DataFrame first, then filter based on parameters.
        base_coef_df = pd.DataFrame({
            "feature": feature_cols,
            "coefficient": coefs,
            "abs_coef": np.abs(coefs) # Check this
        })
        print("After coef-direction filter =", len(base_coef_df.query("coefficient > 0.1")) )
        # MODIFIED: Flexible filtering based on `coef_direction` parameter.
        if coef_direction == 'positive':
            coef_df = base_coef_df.query(f"coefficient > {coef_threshold}").sort_values("coefficient", ascending=False)
        elif coef_direction == 'negative':
            coef_df = base_coef_df.query(f"coefficient < -{coef_threshold}").sort_values("coefficient", ascending=True)
        elif coef_direction == 'all':
            coef_df = base_coef_df.query(f"abs_coef > {coef_threshold}").sort_values("abs_coef", ascending=False)
        else:
            # Default to an empty DataFrame if the direction is invalid
            print(f"ERROR-------adding empty df for {stance_name}")
            coef_df = pd.DataFrame()

        lr_results[stance_name] = coef_df
        summary_lines.append(f"[{stance_name}] Accuracy: {cv_score:.4f}\n")
        for feat, coef in zip(coef_df["feature"], coef_df["coefficient"]):
            summary_lines.append(f"  {feat}: {coef:.4f}\n")
        summary_lines.append("\n")

    # ... (Plotting for LR coefficients remains the same) ...
    # For brevity, omitting the bar plot code which doesn't change.
    print("Rows in global LR before any filtering =", len(lr_results["All Stances"]))
    fig, axes = plt.subplots(2, 2, figsize=(7, 6))
    axes = axes.flatten()
    for i, (stance_name, _) in enumerate(stances):
        if stance_name in lr_results and not lr_results[stance_name].empty:
            coef_df = lr_results[stance_name]
            sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
                        palette="viridis", orient="h")
            axes[i].set_title(stance_name, fontsize=10)
            axes[i].axvline(0, color="gray", linestyle="--")
        else:
            axes[i].axis("off")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    lr_plot_path = f"{output_dir}/lr_{coef_direction}_coef_features_combined.png"
    plt.savefig(lr_plot_path, dpi=300, bbox_inches="tight")
    plt.close()

    # Filter the significance data once for the whole analysis
    sig_filtered_by_source = significance_df[
        (significance_df["dataset"] == dataset) &
        (significance_df["source"] == source)
    ]
    
    
    # MODIFIED: This section now flexibly handles significance filtering.
    all_filtered_features_data = []
    
    
    
#     for stance_name, coef_df in lr_results.items():
#         if not coef_df.empty:
#             # Merge LR results with significance data
#             merged_df = pd.merge(coef_df, sig_filtered_by_source, on="feature", how="left")
            
#             # Apply significance filter if not 'all'
#             if significance_filter in ['yes', 'no']:
#                 # Keep rows that match the filter OR where significance is not available (NaN)
#                 final_stance_df = merged_df[merged_df["significant"].fillna(significance_filter) == significance_filter].copy()
#             else: # significance_filter == 'all'
#                 final_stance_df = merged_df.copy()

#             if not final_stance_df.empty:
#                 final_stance_df['stance_name'] = stance_name
#                 all_filtered_features_data.append(final_stance_df)
#         else:
#             print(f"Printing empty df name ---------{stance_name}")
    
    for stance_name, coef_df in lr_results.items():
        print(f"Working on ------{stance_name}")# stance_name = "All Stances", "AGAINST", …
        if coef_df.empty:
            print(f"PROBLEM ------ {stance_name}")
            continue

        # 1️⃣  pick ONLY the significance rows whose feature appears in this LR slice
        sig_rows = sig_filtered_by_source[
            sig_filtered_by_source["feature"].isin(coef_df["feature"])
        ].copy()

        # 2️⃣  tag every row with the stance we’re processing right now
        #     (this automatically sets "All Stances" when stance_name == "All Stances")
        #sig_rows["stance_name"] = stance_name

        # 3️⃣  merge LR coefficients  +  significance info
        #final_stance_df = coef_df.merge(sig_rows, on="feature", how="left")
        final_stance_df = pd.merge(sig_rows, coef_df, on="feature",how="left" )
        final_stance_df["stance"] = stance_name
        # 4️⃣  apply your significance filter if you want
        if significance_filter in {"yes", "no"}:
            final_stance_df = final_stance_df[
                final_stance_df["significant"].fillna(significance_filter) == significance_filter
            ]
        print(f"Length of {stance_name} df: {len(final_stance_df)}")
        # 5️⃣  store / collect
        if not final_stance_df.empty:
            print(f"Appendig df for {stance_name}")
            #print(df.head())
            all_filtered_features_data.append(final_stance_df)

    # This DataFrame now contains all features that passed the filters for the whole experiment
    if all_filtered_features_data:
        combined_features_df = pd.concat(all_filtered_features_data, ignore_index=True)
        save_path = f"{output_dir}/filtered_features_summary.csv"
        combined_features_df.to_csv(save_path, index=False)
        print(f"[Saved] Filtered features summary for this run to: {save_path}")
    else:
        combined_features_df = pd.DataFrame()

    # Get the unique list of features to perform quartile analysis on
    features_for_quartile_analysis = combined_features_df["feature"].unique().tolist()
    
    summary_lines.append(f"=== Features selected for Quartile Analysis ({len(features_for_quartile_analysis)} total) ===\n")
    for f in features_for_quartile_analysis:
        summary_lines.append(f"  {f}\n")
    summary_lines.append("\n")

    quartile_stats = {}
    # MODIFIED: The loop now iterates over the dynamically created list of features.
    for feature in features_for_quartile_analysis:
        summary_lines.append(f"--- {feature} Quartile Analysis ---\n")
        fig, axes = plt.subplots(2, 2, figsize=(7, 6), sharey=True)
        axes = axes.flatten()
        max_rate = -1
        max_bin = "-"
        for i, (stance_name, stance_val) in enumerate(stances):
            df = df_all.copy()
            if stance_val is not None:
                df = df[df["stance"] == stance_val]
            if feature not in df.columns or df.empty:
                axes[i].axis("off")
                continue
            data = df[[feature, "label"]].dropna()
            try:
                data["quantile"], bins = pd.qcut(data[feature], q=quantiles, retbins=True, duplicates="drop")
            except ValueError:
                axes[i].axis("off")
                continue
            grp = data.groupby(["quantile","label"], observed=True).size().unstack(fill_value=0)
            
            
            if "Correct" not in grp.columns: grp["Correct"] = 0
            if "Misclassified" not in grp.columns: grp["Misclassified"] = 0
            
            grp["Total"] = grp.sum(axis=1)
            grp["Correct Rate"] = grp["Correct"] / grp["Total"]
            grp["Misclassified Rate"] = grp["Misclassified"] / grp["Total"]
            summary_lines.append(f"{stance_name}:\n")
            for q in grp.index:
                mis_rate = grp.loc[q, "Misclassified Rate"]
                if mis_rate > max_rate:
                    max_rate = mis_rate
                    max_bin = str(q)
                    max_stance = stance_name
                summary_lines.append(f"  {q}: Correct={grp.loc[q,'Correct Rate']:.3f}, Mis={mis_rate:.3f}\n")
            long = grp.reset_index().melt(id_vars="quantile",
                                          value_vars=["Correct Rate","Misclassified Rate"],
                                          var_name="Type", value_name="Rate")
            sns.barplot(data=long, x="quantile", y="Rate", hue="Type", ax=axes[i])
            axes[i].set_title(stance_name, fontsize=10)
            axes[i].tick_params(axis='x', labelrotation=45)
            if axes[i].get_legend():
                axes[i].legend_.remove()
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=9)
        plt.tight_layout(rect=[0, 0.08, 1, 0.95])
        save_path = f"{output_dir}/filtered_quartile_{feature}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        summary_lines.append("\n")
        quartile_stats[feature] = {
            "max_misclass_rate": max_rate * 100,
            "max_bin": max_bin,
            "stance": max_stance
        }

    final_lr_results_for_summary = {}
    

    for stance_name, coef_df in lr_results.items():
        if not coef_df.empty:
            # Keep only the rows in the coefficient dataframe where the feature
            # exists in our definitive, fully-filtered list.
            final_df = coef_df[coef_df['feature'].isin(features_for_quartile_analysis)].copy()
            
            if not final_df.empty:
                final_lr_results_for_summary[stance_name] = final_df

    # Create and save summary table
    tables = extract_summary_tables_per_stance(
        final_lr_results_for_summary, combined_features_df, quartile_stats)
    for stance, df in tables.items():
        if not df.empty:
            df.to_csv(f"{output_dir}/summary_table_{stance.lower()}.csv", index=False)
            with open(f"{output_dir}/summary_table_{stance.lower()}.tex", "w") as f:
                f.write(df.to_latex(index=False, float_format="%.2f", escape=True))

    summary_path = f"{output_dir}/analysis_summary.txt"
    with open(summary_path, "w") as f:
        f.writelines(summary_lines)
    print(f"[Saved] Analysis summary to: {summary_path}")
    return lr_results

In [37]:
significance_df = pd.read_csv("all_feature_significance_summary.csv")

dataset ='whole'    # ['covid', 'pstance', 'semeval', 'whole']
source = 'whole'   # ['overall', 'target', 'whole']]

if source == 'whole' and dataset == 'whole':
    print(f"Working on Source: {source} and Dataset: {dataset}")
    correct_path = f"/home/p/parush/style_markers/classifications/{dataset}/correctly_classified_examples_processed.csv"
    misclassified_path = f"/home/p/parush/style_markers/classifications/{dataset}/misclassified_examples_processed.csv"
    print(f"correct_path: {correct_path}")
    print(f"misclassified_path: {misclassified_path}")
else:
    print(f"Working on Source: {source} and Dataset: {dataset}")
    correct_path = f"/home/p/parush/style_markers/classifications/{dataset}/{source}/correctly_classified_examples_processed.csv"
    misclassified_path = f"/home/p/parush/style_markers/classifications/{dataset}/{source}/misclassified_examples_processed.csv"
    print(f"correct_path: {correct_path}")
    print(f"misclassified_path: {correct_path}")
    
experiments_to_run = [
        {'coef': 'positive', 'sig': 'yes'},
        {'coef': 'positive', 'sig': 'no'},
        {'coef': 'negative', 'sig': 'yes'},
        {'coef': 'negative', 'sig': 'no'}
    ]

coef_thresholds = np.round(np.arange(0.10, 1.00, 0.10), 2).tolist()

for thr in coef_thresholds:                                   # ‹thr› = current threshold
    print(f"\n{'#' * 70}")
    print(f"★  STARTING THRESHOLD RUN  →  coef_threshold = {thr:.2f}")
    print(f"{'#' * 70}")

    for run_idx, cfg in enumerate(experiments_to_run, start=1):
        coef_dir = cfg["coef"]          # 'positive' | 'negative' | 'all'
        sig_filt = cfg["sig"]           # 'yes' | 'no' | 'all'

        print(f"\n{'-'*50}")
        print(f"  ▶ Experiment {run_idx:02d}/{len(experiments_to_run)}")
        print(f"     • Coefficient direction : {coef_dir}")
        print(f"     • Significance filter   : {sig_filt}")
        print(f"     • Threshold (|β|)       : {thr:.2f}")
        print(f"{'-'*50}")

        _ = analyze_lr_significance_quartiles_with_filter_and_summary(
                correct_path      = correct_path,
                misclassified_path= misclassified_path,
                significance_df   = significance_df,
                dataset           = dataset,
                source            = source,
                coef_direction    = coef_dir,
                significance_filter = sig_filt,
                coef_threshold    = thr,          # ← current threshold
                quantiles         = 4
        )


# coef_direction = 'all'
# significance_filter = 'yes'
# print(f"\n{'='*60}")
# print(f"  RUNNING EVALUATION FOR EXPERIMENT ")
# print(f"  Coefficient Direction: '{coef_direction}', Significance Filter: '{significance_filter}'")
# print(f"{'='*60}\n")
# lr_results = analyze_lr_significance_quartiles_with_filter_and_summary(
#     correct_path=correct_path,
#     misclassified_path=misclassified_path,
#     significance_df=significance_df,
#     coef_threshold=0.1, 
#     coef_direction=coef_direction,
#     significance_filter=significance_filter,# load this from your significance summary CSV
#     dataset=dataset,
#     source=source,  # or "overall" or "whole"
#     quantiles=4
# )


Working on Source: whole and Dataset: whole
correct_path: /home/p/parush/style_markers/classifications/whole/correctly_classified_examples_processed.csv
misclassified_path: /home/p/parush/style_markers/classifications/whole/misclassified_examples_processed.csv

######################################################################
★  STARTING THRESHOLD RUN  →  coef_threshold = 0.10
######################################################################

--------------------------------------------------
  ▶ Experiment 01/4
     • Coefficient direction : positive
     • Significance filter   : yes
     • Threshold (|β|)       : 0.10
--------------------------------------------------
After coef-direction filter = 12
After coef-direction filter = 9
After coef-direction filter = 12
After coef-direction filter = 12
Rows in global LR before any filtering = 12


/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be 

Working on ------All Stances
Length of All Stances df: 18
Appendig df for All Stances
Working on ------AGAINST
Length of AGAINST df: 11
Appendig df for AGAINST
Working on ------FAVOR
Length of FAVOR df: 15
Appendig df for FAVOR
Working on ------NONE
Length of NONE df: 16
Appendig df for NONE
[Saved] Filtered features summary for this run to: paper_plots_v3/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes/filtered_features_summary.csv
[Saved] Analysis summary to: paper_plots_v3/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes/analysis_summary.txt

--------------------------------------------------
  ▶ Experiment 02/4
     • Coefficient direction : positive
     • Significance filter   : no
     • Threshold (|β|)       : 0.10
--------------------------------------------------
After coef-direction filter = 12
After coef-direction filter = 9
After coef-direction filter = 12
After coef-direction filter = 12
Rows in global LR before any filtering = 12


/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],
/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be 

Working on ------All Stances
Length of All Stances df: 18
Appendig df for All Stances
Working on ------AGAINST
Length of AGAINST df: 16
Appendig df for AGAINST
Working on ------FAVOR
Length of FAVOR df: 21
Appendig df for FAVOR
Working on ------NONE
Length of NONE df: 20
Appendig df for NONE
[Saved] Filtered features summary for this run to: paper_plots_v3/0.1_threshold_experiment/whole_whole_coef-positive_sig-no/filtered_features_summary.csv
[Saved] Analysis summary to: paper_plots_v3/0.1_threshold_experiment/whole_whole_coef-positive_sig-no/analysis_summary.txt

--------------------------------------------------
  ▶ Experiment 03/4
     • Coefficient direction : negative
     • Significance filter   : yes
     • Threshold (|β|)       : 0.10
--------------------------------------------------
After coef-direction filter = 12
After coef-direction filter = 9
After coef-direction filter = 12
After coef-direction filter = 12
Rows in global LR before any filtering = 9


...

/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],


Working on ------All Stances
PROBLEM ------ All Stances
Working on ------AGAINST
PROBLEM ------ AGAINST
Working on ------FAVOR
PROBLEM ------ FAVOR
Working on ------NONE
Length of NONE df: 1
Appendig df for NONE
[Saved] Filtered features summary for this run to: paper_plots_v3/0.9_threshold_experiment/whole_whole_coef-positive_sig-yes/filtered_features_summary.csv
[Saved] Analysis summary to: paper_plots_v3/0.9_threshold_experiment/whole_whole_coef-positive_sig-yes/analysis_summary.txt

--------------------------------------------------
  ▶ Experiment 02/4
     • Coefficient direction : positive
     • Significance filter   : no
     • Threshold (|β|)       : 0.90
--------------------------------------------------
After coef-direction filter = 12
After coef-direction filter = 9
After coef-direction filter = 12
After coef-direction filter = 12
Rows in global LR before any filtering = 0


/tmp/ipykernel_848942/32697800.py:188: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="coefficient", y="feature", data=coef_df, ax=axes[i],


Working on ------All Stances
PROBLEM ------ All Stances
Working on ------AGAINST
PROBLEM ------ AGAINST
Working on ------FAVOR
PROBLEM ------ FAVOR
Working on ------NONE
Length of NONE df: 2
Appendig df for NONE
[Saved] Filtered features summary for this run to: paper_plots_v3/0.9_threshold_experiment/whole_whole_coef-positive_sig-no/filtered_features_summary.csv
[Saved] Analysis summary to: paper_plots_v3/0.9_threshold_experiment/whole_whole_coef-positive_sig-no/analysis_summary.txt

--------------------------------------------------
  ▶ Experiment 03/4
     • Coefficient direction : negative
     • Significance filter   : yes
     • Threshold (|β|)       : 0.90
--------------------------------------------------
After coef-direction filter = 12
After coef-direction filter = 9
After coef-direction filter = 12
After coef-direction filter = 12
Rows in global LR before any filtering = 0
Working on ------All Stances
PROBLEM ------ All Stances
Working on ------AGAINST
PROBLEM ------ AGAINST

KeyError: 'feature'